# MAPLE Integration Test

This notebook checks that the R-based MAPLE workflow can be run from the Python project environment and that its outputs can be read back into Python for downstream extremal search.

The goal is to understand the MAPLE pipeline, verify R/Python handoff, and inspect the objects produced by the simulated annealing reconstruction procedure.

## MAPLE File Roles

The MAPLE folder contains the following core files:

- `simdata.R`: generates synthetic survival data with true hidden subgroup labels.
- `target.R`: computes target statistics from a proposed label assignment, including subgroup hazard ratios and median survival summaries.
- `constraints_target.R`: constructs count constraints and target statistics from the full data.
- `loss.R`: defines the relative worst-case loss between estimated and target statistics.
- `sa.R`: implements simulated annealing over feasible label assignments.
- `main.R`: runs the full workflow.

The key MAPLE output is a reconstructed label vector that minimizes discrepancy between candidate-label summaries and target summaries while satisfying count constraints.

In [3]:
from pathlib import Path
import subprocess
import pandas as pd
import numpy as np

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

MAPLE_DIR = PROJECT_ROOT / "resolve-ipd" / "maple"
OUTPUT_DIR = PROJECT_ROOT / "results" / "maple_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("MAPLE directory:", MAPLE_DIR)
print("Output directory:", OUTPUT_DIR)

assert MAPLE_DIR.exists(), "MAPLE directory not found"

Project root: /Users/jonathanma/Desktop/DS Projects/covar-recon-IPD
MAPLE directory: /Users/jonathanma/Desktop/DS Projects/covar-recon-IPD/resolve-ipd/maple
Output directory: /Users/jonathanma/Desktop/DS Projects/covar-recon-IPD/results/maple_outputs


In [4]:
# Verifying that Radian works
cmd = ["Rscript", "-e", "cat('Rscript works\\n')"]

result = subprocess.run(
    cmd,
    capture_output=True,
    text=True
)

print(result.stdout)
print(result.stderr)
assert result.returncode == 0

Rscript works




## Create MAPLE Test Runner

This temporary R script runs the MAPLE simulation and saves the generated data, true labels, reconstructed labels, and summary diagnostics to CSV files.

In [10]:
runner_path = OUTPUT_DIR / "run_maple_test.R"

runner_code = f"""
library(survival)
library(dplyr)
library(here)

setwd("{PROJECT_ROOT}")

source(file.path("resolve-ipd", "maple", "target.R"))
source(file.path("resolve-ipd", "maple", "loss.R"))
source(file.path("resolve-ipd", "maple", "sa.R"))
source(file.path("resolve-ipd", "maple", "simdata.R"))

N <- 400
n_g0 <- 200

df_full <- generate_simdata(N = N, n_g0 = n_g0, seed = 0)

source(file.path("resolve-ipd", "MAPLE", "constraints_target.R"))

df_full$locked_label <- NA
df_full$locked <- NA
df_full$max <- NA

for (lbl in unique(df_full$label)) {{
  for (z in unique(df_full$Z)) {{
    idx_subset <- which(df_full$label == lbl & df_full$Z == z)
    times_subset <- df_full$time[idx_subset]
    max_time <- max(times_subset)
    idx_max_time <- idx_subset[which(times_subset == max_time)]
    df_full$locked_label[idx_max_time] <- lbl
    df_full$locked[idx_max_time] <- TRUE
    df_full$max[idx_max_time] <- TRUE
  }}
}}

res_sa <- simulated_annealing_swap_multi(
  df_hidden      = df_full,
  target_stats   = target_stats,
  constraints    = constraints,
  seeds = c(2),
  loss_fun = compute_rel_worst_loss,
  target_fn = compute_summary_stat_simdata,
  T0 = 1.0,
  cooling = 0.998,
  min_iter = 1000,
  max_iter = 3000,
  stagnation_stop_iter = 500,
  verbose = TRUE
)

df_out <- df_full
df_out$maple_label <- as.integer(res_sa$best_labels)

write.csv(df_out, file.path("{OUTPUT_DIR}", "maple_test_output.csv"), row.names = FALSE)

summary_out <- data.frame(
  best_loss = res_sa$best_loss,
  best_seed = res_sa$best_seed
)

write.csv(summary_out, file.path("{OUTPUT_DIR}", "maple_test_summary.csv"), row.names = FALSE)

constraint_out <- data.frame(
  constraint = names(constraints),
  value = as.numeric(unlist(constraints))
)

write.csv(constraint_out, file.path("{OUTPUT_DIR}", "maple_test_constraints.csv"), row.names = FALSE)
"""

runner_path.write_text(runner_code)

print("Wrote runner:", runner_path)

Wrote runner: /Users/jonathanma/Desktop/DS Projects/covar-recon-IPD/results/maple_outputs/run_maple_test.R


In [11]:
result = subprocess.run(
    ["Rscript", str(runner_path)],
    capture_output=True,
    text=True,
    cwd=PROJECT_ROOT
)

print("STDOUT:")
print(result.stdout)

print("STDERR:")
print(result.stderr)

print("Return code:", result.returncode)
assert result.returncode == 0

STDOUT:

STDERR:

Attaching package: ‘dplyr’

The following objects are masked from ‘package:stats’:

    filter, lag

The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union

here() starts at /Users/jonathanma/Desktop/DS Projects/covar-recon-IPD
Running SA across 1 seeds...
[seed=2] starting
[it=1000] T=0.1351 cur=0.3235 best=0.1807 acc=987
[it=1500] no improvement for 500 iterations after min_iter -> stopping
[seed=2] best_loss: 0.180723
Best across seeds: seed=2 best_loss=0.180723

Return code: 0


In [12]:
# load  MAPLE outputs
maple_df = pd.read_csv(OUTPUT_DIR / "maple_test_output.csv")
maple_summary = pd.read_csv(OUTPUT_DIR / "maple_test_summary.csv")
maple_constraints = pd.read_csv(OUTPUT_DIR / "maple_test_constraints.csv")

display(maple_df.head())
display(maple_summary)
display(maple_constraints)

,time,event,Z,label,locked_label,locked,max,maple_label
0,1.757999,1,0,0,NaN,NaN,NaN,1
1,18.378767,0,1,1,NaN,NaN,NaN,1
2,2.339394,1,0,1,NaN,NaN,NaN,0
3,13.610037,0,0,0,NaN,NaN,NaN,1
4,20.247962,0,1,1,NaN,NaN,NaN,1


,best_loss,best_seed
0,0.180723,2


,constraint,value
0,n_0_z1_ev,77
1,n_0_z1,102
2,n_0_z0_ev,69
3,n_0_z0,98
4,n_1_z1_ev,51
5,n_1_z1,82
6,n_1_z0_ev,92
7,n_1_z0,118


In [13]:
pd.crosstab(
    maple_df["label"],
    maple_df["maple_label"],
    rownames=["True label"],
    colnames=["MAPLE label"],
    margins=True
)

MAPLE label,0,1,All
True label,,,
0,109,91,200
1,91,109,200
All,200,200,400


In [14]:
label_accuracy = (maple_df["label"] == maple_df["maple_label"]).mean()
print(f"Label accuracy: {label_accuracy:.3f}")

Label accuracy: 0.545


In [15]:
def compute_maple_style_counts(df, label_col):
    out = {}

    for g in [0, 1]:
        for z in [0, 1]:
            mask = (df[label_col] == g) & (df["Z"] == z)
            out[f"n_{g}_z{z}"] = int(mask.sum())
            out[f"n_{g}_z{z}_ev"] = int((mask & (df["event"] == 1)).sum())

    return out

true_counts = compute_maple_style_counts(maple_df, "label")
maple_counts = compute_maple_style_counts(maple_df, "maple_label")

constraint_check = pd.DataFrame({
    "constraint": true_counts.keys(),
    "target": true_counts.values(),
    "maple": [maple_counts[k] for k in true_counts.keys()],
})

constraint_check["match"] = constraint_check["target"] == constraint_check["maple"]

constraint_check

,constraint,target,maple,match
0,n_0_z0,98,98,True
1,n_0_z0_ev,69,69,True
2,n_0_z1,102,102,True
3,n_0_z1_ev,77,77,True
4,n_1_z0,118,118,True
5,n_1_z0_ev,92,92,True
6,n_1_z1,82,82,True
7,n_1_z1_ev,51,51,True


In [16]:
print("All constraints satisfied:", constraint_check["match"].all())

All constraints satisfied: True


## MAPLE Objective

MAPLE does not optimize label accuracy directly. It searches for a feasible label assignment whose summary statistics match the target summaries.

The loss function used here is a relative worst-case loss:

`max over target statistics of relative absolute error`

Therefore, a reconstruction can satisfy published constraints and summary targets without necessarily recovering the exact true label vector.

In [17]:
handoff_df = maple_df[
    ["time", "event", "Z", "label", "maple_label"]
].copy()

handoff_path = OUTPUT_DIR / "maple_python_handoff.csv"
handoff_df.to_csv(handoff_path, index=False)

print("Saved:", handoff_path)

# IMPORTANT
# MAPLE Z      = treatment
# MAPLE label  = subgroup label

Saved: /Users/jonathanma/Desktop/DS Projects/covar-recon-IPD/results/maple_outputs/maple_python_handoff.csv
